# 01 - Check Python packages and the Windows driver

Identify the active kernel, load the packaged libusb backend, and optionally inspect
Windows driver bindings. These checks do not claim a motor interface or send APT commands.
Driver changes are a separate operator action; nothing here installs or replaces a driver.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


## Check the active kernel and packages

The library requires Python 3.12 or later, PyUSB, and libusb-package. This USB implementation
uses WinUSB; Kinesis, pythonnet, a COM port, and Thorlabs SDK DLLs are not required.
If a package is missing, stop here and select the kernel described in the index.


In [ ]:
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
assert sys.version_info >= (3, 12), "Use Python 3.12 or later."
for package in ("k10cr2-controller", "pyusb", "libusb-package", "ipykernel"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"MISSING: {package}; check installation and the selected kernel")

## Load libusb without enumerating devices

A loaded backend establishes DLL availability, not the mount's driver binding or access.
A missing backend is a Python/backend installation problem to resolve before discovery.


In [ ]:
import libusb_package

backend = libusb_package.get_libusb1_backend()
if backend is None:
    raise RuntimeError("Packaged libusb backend unavailable; check the Python environment.")
print("libusb backend loaded:", type(backend).__name__)

## Optional Windows PnP inspection

Set `CHECK_WINDOWS_DRIVER = True` to run read-only PowerShell queries. Candidates match
VID `0403` / PID `FAF0`; these IDs alone do not identify a K10CR2. Check the name and serial.
For the intended device, expect service `WinUSB` and PnP status `OK`. INF filenames vary
by computer. If access is restricted, inspect the device's Driver Details and Hardware IDs
in Device Manager. Do not change other FTDI adapters' bindings.


In [ ]:
import json
import subprocess

CHECK_WINDOWS_DRIVER = False
DRIVER_QUERY = r"""
$ErrorActionPreference = 'Stop'
$devices = @(Get-PnpDevice -PresentOnly | Where-Object {
    $_.InstanceId -like 'USB\VID_0403&PID_FAF0*'
})
$rows = @(foreach ($device in $devices) {
    $serviceKey = 'DEVPKEY_Device_Service'
    $service = Get-PnpDeviceProperty -InstanceId $device.InstanceId -KeyName $serviceKey
    $infKey = 'DEVPKEY_Device_DriverInfPath'
    $inf = Get-PnpDeviceProperty -InstanceId $device.InstanceId -KeyName $infKey
    [pscustomobject]@{
        name = $device.FriendlyName
        instance_id = $device.InstanceId
        status = [string]$device.Status
        service = $service.Data
        driver_inf = $inf.Data
    }
})
ConvertTo-Json -InputObject $rows -Depth 4 -Compress
"""

if CHECK_WINDOWS_DRIVER:
    if sys.platform != "win32":
        raise RuntimeError("This driver check is for Windows only.")
    result = subprocess.run(
        ["powershell.exe", "-NoProfile", "-NonInteractive", "-Command", DRIVER_QUERY],
        capture_output=True,
        text=True,
        check=True,
        timeout=30,
    )
    driver_rows = json.loads(result.stdout)
    print(json.dumps(driver_rows, indent=2))
    if not driver_rows:
        print("No matching present PnP device: check power, cable, and Device Manager.")
else:
    print("Skipped Windows driver inspection; set CHECK_WINDOWS_DRIVER = True to run it.")

Next: [02 - Device discovery](02_device_discovery.ipynb). A correct binding still
requires a successful discovery and identity check. See also the repository's
[USB bridge contract](../../docs/commands.md#usb-bridge-setup-not-apt).
